# BARAM 2026 — LB 하락 진단: 후처리 연도 견고성 재검증 + CatBoost

**배경**: v5 실전 0.61206 (홀드아웃 0.6461 대비 −0.034). FICR −0.053이 주범.
가설: ① FICR nudge(scale 최대 1.15)가 2024엔 맞았지만 2025로 이전 실패(과대 상향), ② 2025가 어려운 해.

**이 노트북이 답할 것**
1. 후처리 후보를 **2023·2024 두 폴드 모두**에서 평가 → 연도 견고한 선택은 무엇인가 (기존엔 2024만 봄)
2. nudge 전후 **예측 편향(평균 CF)** — 상향 편향이 실재하는가
3. **CatBoost** 3번째 앙상블 멤버 — GBM 계열 교체/추가의 실익은 얼마인가

속도를 위해 MLP는 시드 3개(진단엔 충분). 최종 재구축은 별도.

## 0. 설정

In [1]:
import os
os.environ.setdefault("OMP_NUM_THREADS","1")
import warnings; warnings.filterwarnings("ignore")
import json, numpy as np, pandas as pd
import torch, torch.nn as nn
torch.set_num_threads(1)
import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingRegressor as HGB
from catboost import CatBoostRegressor
import wind_lib as W
from official_metric import group_scores

DEV="mps" if torch.backends.mps.is_available() else "cpu"
SEEDS=[15,0,1]; BLEND=0.7
MLP_CFG=dict(h=256,depth=3,drop=0.0,lr=0.0015868563457489381,wd=1e-4,bs=256,emb=4)
GROUPS=(1,2,3)
FR,TGT={},{}
for g in GROUPS:
    df,tgt=W.load_train(g); TGT[g]=tgt
    FR[g]=W.add_spatial(W.build(df,g),"train")
BASE_ALL=[c for c in W.feature_cols(FR[1]) if c not in W.SPATIAL_COLS]+["pc_pred_cf"]
FEATS=W.lean_features(BASE_ALL)+W.SPATIAL_COLS
print("device:",DEV,"| feats:",len(FEATS))

def lgbm(): return lgb.LGBMRegressor(objective="mae",n_estimators=600,learning_rate=0.03,num_leaves=63,
    min_child_samples=60,subsample=0.8,subsample_freq=1,colsample_bytree=0.7,reg_lambda=1.0,
    random_state=42,n_jobs=1,verbose=-1)
def histgbm(): return HGB(loss="absolute_error",max_iter=600,learning_rate=0.03,max_leaf_nodes=63,
    l2_regularization=1.0,random_state=42)
def catb(): return CatBoostRegressor(loss_function="MAE",iterations=800,learning_rate=0.05,
    depth=8,l2_leaf_reg=3.0,random_seed=42,verbose=0,thread_count=1)

class MLP(nn.Module):
    def __init__(s,nf,ng=3,h=256,depth=3,drop=0.0,emb=4):
        super().__init__(); s.emb=nn.Embedding(ng,emb)
        L=[nn.Linear(nf+emb,h),nn.GELU(),nn.Dropout(drop)]
        for _ in range(depth-1): L+=[nn.Linear(h,h),nn.GELU(),nn.Dropout(drop)]
        L+=[nn.Linear(h,1)]; s.net=nn.Sequential(*L)
    def forward(s,x,g): return s.net(torch.cat([x,s.emb(g)],-1)).squeeze(-1)

def train_one(pool_tr,seed):
    torch.manual_seed(seed); np.random.seed(seed)
    pool_tr=pool_tr.sort_values("kst_dtm")
    mu=pool_tr[FEATS].mean(); sd=pool_tr[FEATS].std()+1e-8
    X=((pool_tr[FEATS]-mu)/sd).to_numpy(np.float32)
    y=pool_tr["cf"].to_numpy(np.float32); gid=pool_tr["gid"].to_numpy(np.int64)
    n=len(X); cut=int(n*0.85)
    Xt=torch.tensor(X,device=DEV); yt=torch.tensor(y,device=DEV); gt=torch.tensor(gid,device=DEV)
    net=MLP(len(FEATS),**{k:MLP_CFG[k] for k in ("h","depth","drop","emb")}).to(DEV)
    opt=torch.optim.AdamW(net.parameters(),lr=MLP_CFG["lr"],weight_decay=MLP_CFG["wd"])
    best=1e9; st=None; pat=0
    for ep in range(120):
        net.train(); perm=np.random.permutation(np.arange(cut))
        for i in range(0,len(perm),MLP_CFG["bs"]):
            b=torch.tensor(perm[i:i+MLP_CFG["bs"]],device=DEV)
            opt.zero_grad(); ((net(Xt[b],gt[b])-yt[b]).abs().mean()).backward(); opt.step()
        net.eval()
        with torch.no_grad(): vl=(net(Xt[cut:],gt[cut:])-yt[cut:]).abs().mean().item()
        if vl<best-1e-5: best=vl; st={k:v.clone() for k,v in net.state_dict().items()}; pat=0
        else: pat+=1
        if pat>=10: break
    net.load_state_dict(st); return net,(mu,sd)

def predict_one(net,scaler,fr,g,cap):
    mu,sd=scaler
    X=torch.tensor(((fr[FEATS]-mu)/sd).to_numpy(np.float32),device=DEV)
    gid=torch.full((len(fr),),g-1,dtype=torch.long,device=DEV)
    net.eval()
    with torch.no_grad(): p=net(X,gid).cpu().numpy()
    return np.clip(p,0,1)*cap

def blend_predict(tr_frames, extra_cat=False):
    """{g:(train,predict)} -> {g: pred}. extra_cat=True면 GBM에 CatBoost 포함(3모델 평균)."""
    pool=[]
    for g,(tr2,_) in tr_frames.items():
        p=tr2[FEATS+["kst_dtm"]].copy(); p["cf"]=tr2[TGT[g]]/W.CAP[g]; p["gid"]=g-1; pool.append(p)
    pool=pd.concat(pool,ignore_index=True)
    mlp_acc={g:[] for g in tr_frames}
    for sd_ in SEEDS:
        net,scaler=train_one(pool,sd_)
        for g,(_,va2) in tr_frames.items():
            mlp_acc[g].append(predict_one(net,scaler,va2,g,W.CAP[g]))
    out={}
    for g,(tr2,va2) in tr_frames.items():
        cap=W.CAP[g]; tgt=TGT[g]
        preds=[lgbm().fit(tr2[FEATS],tr2[tgt]).predict(va2[FEATS]),
               histgbm().fit(tr2[FEATS].to_numpy(),tr2[tgt].to_numpy()).predict(va2[FEATS].to_numpy())]
        if extra_cat:
            preds.append(catb().fit(tr2[FEATS],tr2[tgt]).predict(va2[FEATS]))
        gp=np.clip(np.mean(preds,axis=0),0,cap)
        mp=np.mean(mlp_acc[g],axis=0)
        out[g]=np.clip((1-BLEND)*gp+BLEND*mp,0,cap)
    return out

device: mps | feats: 65


## 1. 폴드별 blend 예측 + 폴드 내부 OOF (후처리 fit용)

In [2]:
FOLDS={2023:[2022],2024:[2022,2023]}   # val_year -> train_years
DATA={}   # vy -> dict(g -> dict(tr2,va2,pred,oof))
for vy,tys in FOLDS.items():
    ent={}
    for g in GROUPS:
        tgt=TGT[g]; cap=W.CAP[g]; fr=FR[g]; yr=fr.kst_dtm.dt.year
        tr=fr[yr.isin(tys)]; va=fr[yr==vy]
        if len(tr)==0 or len(va)==0: continue
        iso=W.fit_powercurve(tr,tgt,cap)
        ent[g]=dict(tr2=W.with_pc(tr,iso),va2=W.with_pc(va,iso))
    # val 예측
    pred=blend_predict({g:(e["tr2"],e["va2"]) for g,e in ent.items()})
    for g in ent: ent[g]["pred"]=pred[g]
    # 폴드 내부 OOF: train_years 2개면 연도 swap, 1개면 시간순 70/30
    for g,e in ent.items():
        tr2=e["tr2"]; oof=np.full(len(tr2),np.nan)
        yrs=sorted(tr2.kst_dtm.dt.year.unique())
        if len(yrs)>=2:
            for ty in yrs:
                m_in=tr2.kst_dtm.dt.year!=ty; m_out=(tr2.kst_dtm.dt.year==ty).to_numpy()
                oof[m_out]=blend_predict({g:(tr2[m_in],tr2[tr2.kst_dtm.dt.year==ty])})[g]
        else:
            n=len(tr2); cut=int(n*0.7)
            oof[cut:]=blend_predict({g:(tr2.iloc[:cut],tr2.iloc[cut:])})[g]
        e["oof"]=oof
    DATA[vy]=ent
    print(f"fold →{vy} 준비 완료 (groups {list(ent)})")

fold →2023 준비 완료 (groups [1, 2])


fold →2024 준비 완료 (groups [1, 2, 3])


## 2. 후처리 후보를 두 해 모두에서 평가

In [3]:
def debias_fit(tr,tgt,oof):
    d=tr.assign(oof=oof); d=d[np.isfinite(d["oof"])].copy(); d["resid"]=d[tgt]-d["oof"]
    d["lb"]=pd.cut(d["lead_h"],bins=[15,21,27,33,40],labels=False,include_lowest=True)
    d["wq"]=pd.qcut(d["gfs_wind_speed_100m_mean"],5,labels=False,duplicates="drop")
    return d.groupby(["lb","wq"])["resid"].mean(), d["resid"].mean()
def debias_apply(va,pred,tbl,glob):
    v=va.copy()
    v["lb"]=pd.cut(v["lead_h"],bins=[15,21,27,33,40],labels=False,include_lowest=True)
    v["wq"]=pd.qcut(v["gfs_wind_speed_100m_mean"],5,labels=False,duplicates="drop")
    return pred+np.array([tbl.get(k,glob) for k in zip(v["lb"],v["wq"])])
def nudge_fit(tr,tgt,oof,cap,smax=1.15,shmax=0.06):
    d=tr.assign(oof=oof); d=d[np.isfinite(d["oof"])]
    yt=d[tgt].to_numpy(); yp=d["oof"].to_numpy(); best=(1.0,0.0); bf=-1
    for s in np.linspace(2-smax,smax,26):
        for sh in np.linspace(-shmax,shmax,25)*cap:
            f=group_scores(yt,np.clip(yp*s+sh,0,cap),cap)[1]
            if f>bf: bf=f; best=(s,sh)
    return best

POSTS=["P0_none","P1_debias","P3_nudge","P3c_nudge_cons","P5_deb_nudge","P5c_deb_nudge_cons"]
rows=[]; bias_rows=[]
for vy,ent in DATA.items():
    for g,e in ent.items():
        tgt=TGT[g]; cap=W.CAP[g]; tr2=e["tr2"]; va2=e["va2"]; bp=e["pred"]; oof=e["oof"]
        tbl,glob=debias_fit(tr2,tgt,oof)
        s_full=nudge_fit(tr2,tgt,oof,cap)                       # 기존: scale<=1.15
        s_cons=nudge_fit(tr2,tgt,oof,cap,smax=1.05,shmax=0.02)  # 보수: scale<=1.05, shift<=2%
        p1=np.clip(debias_apply(va2,bp,tbl,glob),0,cap)
        cand={"P0_none":bp,"P1_debias":p1,
              "P3_nudge":np.clip(bp*s_full[0]+s_full[1],0,cap),
              "P3c_nudge_cons":np.clip(bp*s_cons[0]+s_cons[1],0,cap),
              "P5_deb_nudge":np.clip(p1*s_full[0]+s_full[1],0,cap),
              "P5c_deb_nudge_cons":np.clip(p1*s_cons[0]+s_cons[1],0,cap)}
        yt=va2[tgt].to_numpy()
        for name,p in cand.items():
            nm,fi=group_scores(yt,p,cap)
            rows.append(dict(fold=vy,group=g,post=name,nmae=nm,ficr=fi,contrib=fi-nm))
        # 편향 진단
        bias_rows.append(dict(fold=vy,group=g,
            actual_cf=100*yt.mean()/cap, raw_cf=100*bp.mean()/cap,
            nudged_cf=100*cand["P5_deb_nudge"].mean()/cap,
            cons_cf=100*cand["P5c_deb_nudge_cons"].mean()/cap,
            nudge_scale=s_full[0], cons_scale=s_cons[0]))
pp=pd.DataFrame(rows)

def fold_total(vy,post):
    s=pp[(pp.fold==vy)&(pp.post==post)]
    return 0.5*(1-s.nmae.mean())+0.5*s.ficr.mean()
print("=== 후처리별 공식 총점 (두 해) ===")
res={}
for p in POSTS:
    t23,t24=fold_total(2023,p),fold_total(2024,p)
    res[p]=(t23,t24)
    print(f"  {p:22s}: 2023={t23:.4f}  2024={t24:.4f}  min={min(t23,t24):.4f}  mean={(t23+t24)/2:.4f}")
# 견고 선택: worst-year 최대
robust=max(POSTS,key=lambda p:min(res[p]))
print(f"\n견고(worst-year) 선택: {robust}")
print(f"기존 방식(2024만 보고 선택)이 골랐을 것: {max(POSTS,key=lambda p:res[p][1])}")

=== 후처리별 공식 총점 (두 해) ===
  P0_none               : 2023=0.5992  2024=0.6079  min=0.5992  mean=0.6035
  P1_debias             : 2023=0.6010  2024=0.6006  min=0.6006  mean=0.6008
  P3_nudge              : 2023=0.5972  2024=0.6457  min=0.5972  mean=0.6214
  P3c_nudge_cons        : 2023=0.6125  2024=0.6294  min=0.6125  mean=0.6209
  P5_deb_nudge          : 2023=0.5296  2024=0.6448  min=0.5296  mean=0.5872
  P5c_deb_nudge_cons    : 2023=0.5811  2024=0.6276  min=0.5811  mean=0.6043

견고(worst-year) 선택: P3c_nudge_cons
기존 방식(2024만 보고 선택)이 골랐을 것: P3_nudge


In [4]:
bias=pd.DataFrame(bias_rows)
print("=== 예측 편향 진단 (평균 CF %) ===")
print(bias.round(2).to_string(index=False))
print("\n읽는 법: nudged_cf가 actual_cf보다 크게 높으면 상향 편향 — 2025 FICR 붕괴 가설의 직접 증거.")

=== 예측 편향 진단 (평균 CF %) ===
 fold  group  actual_cf  raw_cf  nudged_cf  cons_cf  nudge_scale  cons_scale
 2023      1      32.41   29.73      46.96    39.61         1.15        1.04
 2023      2      33.45   34.56      50.52    46.08         1.09        1.05
 2024      1      29.18   28.12      34.79    31.69         1.02        1.05
 2024      2      30.04   30.95      36.16    33.84         0.98        1.04
 2024      3      26.13   24.98      34.07    28.18         1.15        1.03

읽는 법: nudged_cf가 actual_cf보다 크게 높으면 상향 편향 — 2025 FICR 붕괴 가설의 직접 증거.


## 3. CatBoost — 3번째 GBM 멤버 실익

In [5]:
cat_rows=[]
for vy,ent in DATA.items():
    pred_cat=blend_predict({g:(e["tr2"],e["va2"]) for g,e in ent.items()}, extra_cat=True)
    for g,e in ent.items():
        yt=e["va2"][TGT[g]].to_numpy(); cap=W.CAP[g]
        nm0,fi0=group_scores(yt,e["pred"],cap)          # LGBM+HistGBM
        nm1,fi1=group_scores(yt,pred_cat[g],cap)        # +CatBoost
        cat_rows.append(dict(fold=vy,group=g,base_contrib=fi0-nm0,cat_contrib=fi1-nm1))
cat=pd.DataFrame(cat_rows)
for vy in DATA:
    s=cat[cat.fold==vy]
    print(f"fold →{vy}: base(2GBM+MLP) contrib평균={s.base_contrib.mean():.4f}  +CatBoost={s.cat_contrib.mean():.4f}  Δ={(s.cat_contrib-s.base_contrib).mean():+.4f}")

fold →2023: base(2GBM+MLP) contrib평균=0.1983  +CatBoost=0.1982  Δ=-0.0001
fold →2024: base(2GBM+MLP) contrib평균=0.2158  +CatBoost=0.2138  Δ=-0.0019


## 4. 결론 요약

In [6]:
summary=dict(
    postproc_totals={p:{"2023":round(res[p][0],4),"2024":round(res[p][1],4)} for p in POSTS},
    robust_choice=robust,
    old_choice_2024_only=max(POSTS,key=lambda p:res[p][1]),
    bias=bias.round(2).to_dict("records"),
    catboost_delta_by_fold={str(vy):round(float((cat[cat.fold==vy].cat_contrib-cat[cat.fold==vy].base_contrib).mean()),4) for vy in DATA})
json.dump(summary,open("submission/ver_6/diagnosis_summary.json","w"),ensure_ascii=False,indent=2)
print(json.dumps(summary,ensure_ascii=False,indent=2))

{
  "postproc_totals": {
    "P0_none": {
      "2023": 0.5992,
      "2024": 0.6079
    },
    "P1_debias": {
      "2023": 0.601,
      "2024": 0.6006
    },
    "P3_nudge": {
      "2023": 0.5972,
      "2024": 0.6457
    },
    "P3c_nudge_cons": {
      "2023": 0.6125,
      "2024": 0.6294
    },
    "P5_deb_nudge": {
      "2023": 0.5296,
      "2024": 0.6448
    },
    "P5c_deb_nudge_cons": {
      "2023": 0.5811,
      "2024": 0.6276
    }
  },
  "robust_choice": "P3c_nudge_cons",
  "old_choice_2024_only": "P3_nudge",
  "bias": [
    {
      "fold": 2023,
      "group": 1,
      "actual_cf": 32.41,
      "raw_cf": 29.73,
      "nudged_cf": 46.96,
      "cons_cf": 39.61,
      "nudge_scale": 1.15,
      "cons_scale": 1.04
    },
    {
      "fold": 2023,
      "group": 2,
      "actual_cf": 33.45,
      "raw_cf": 34.56,
      "nudged_cf": 50.52,
      "cons_cf": 46.08,
      "nudge_scale": 1.09,
      "cons_scale": 1.05
    },
    {
      "fold": 2024,
      "group": 1,
      "ac